### Import Library
Impor semua library yang dibutuhkan untuk data preprocessing, pelatihan model, evaluasi, dan visualisasi.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

### Load Dataset
Dataset dibaca dari file CSV dan nilai-nilai yang kosong diisi dengan median dari masing-masing kolom numerik.

In [ ]:
df = pd.read_csv('model/water_potability.csv')
df.fillna(df.median(numeric_only=True), inplace=True)

### Pisahkan Fitur dan Label
Memisahkan fitur (X) dan label target (y) dari dataset.

In [ ]:
X = df.drop('Potability', axis=1)
y = df['Potability']

### Standardisasi Fitur
Melakukan normalisasi fitur menggunakan StandardScaler agar memiliki skala yang seragam.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Penanganan Imbalanced Dataset
Menggunakan SMOTE untuk menyeimbangkan jumlah data pada masing-masing kelas.

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

### Split Data
Membagi data yang telah diseimbangkan menjadi data latih dan data uji.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

### Training Model + Grid Search
Melatih model Random Forest dan melakukan tuning hyperparameter menggunakan GridSearchCV.

In [ ]:
params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(rf, params, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

### Prediksi
Melakukan prediksi terhadap data uji, baik dalam bentuk kelas maupun probabilitas.

In [ ]:
y_pred = grid_search.predict(X_test)
y_proba = grid_search.predict_proba(X_test)[:, 1]

### Visualisasi ROC Curve
Menghitung dan memvisualisasikan ROC curve serta menghitung nilai AUC dari model.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('static/img/roc_curve.png')
plt.close()

### Visualisasi Feature Importance
Menampilkan seberapa penting setiap fitur dalam proses prediksi menggunakan Random Forest.

In [ ]:
importances = grid_search.best_estimator_.feature_importances_
feature_names = X.columns

plt.figure(figsize=(8, 6))
sns.barplot(x=importances, y=feature_names)
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Fitur')
plt.tight_layout()
plt.savefig('static/img/feature_importance.png')
plt.close()

### Evaluasi Model
Menghitung metrik evaluasi seperti akurasi, presisi, recall, dan F1-score lalu menyimpannya ke dalam file JSON.

In [ ]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred)
}

with open('static/metrics.json', 'w') as f:
    json.dump(metrics, f)

### Confusion Matrix
Menampilkan confusion matrix untuk melihat performa prediksi benar dan salah dari model.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Tidak Layak", "Layak"],
            yticklabels=["Tidak Layak", "Layak"])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('static/img/confusion_matrix.png')
plt.close()

### Menampilkan Hasil Evaluasi
Menampilkan classification report dan confusion matrix di konsol/log.

In [ ]:
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

### Simpan Model dan Scaler
Menyimpan model dan scaler yang telah dilatih ke dalam file agar bisa digunakan kembali di aplikasi web.

In [ ]:
joblib.dump(grid_search.best_estimator_, 'model/final_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')